In [1]:
import os
import sys

from libs.multilevel_squeme.coarsening import coarsen_graph
from libs.multilevel_squeme.driver import k_way_partition
from libs.multilevel_squeme import partition_graph_metis

from libs.utils import (
    load_mtx,
    matrix_to_graph,
    summarize_generic
)

In [2]:
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

k_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_k.mtx")
m_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_m.mtx")
print("Using data_dir:", data_dir)

Using data_dir: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/data


In [3]:
# Load matrices
A_K = load_mtx(k_matrix_path, 'K')
A_M = load_mtx(m_matrix_path, 'M')

if A_K is None or A_M is None:
    raise RuntimeError("Matrix load failed; check file paths printed above.")

print(f"Loaded: K | shape={A_K.shape}, nnz={A_K.nnz}")
print(f"Loaded: M | shape={A_M.shape}, nnz={A_M.nnz}")

Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098
Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098


In [4]:
# Build graphs from matrices
diag_K = A_K.diagonal()

# Map the matrix into the graph (only the K matrix is needed since we are partitioning most based on connectivity)
G_K = matrix_to_graph(A_K, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_K)

print(f"K: |V|={G_K.number_of_nodes()}, |E|={G_K.number_of_edges()}")

K: |V|=960, |E|=14661


In [5]:
coarsen_limit = 80
max_levels = 25
weight = 'weight'

def coarsen_chain(H, trial_seed: int | None):
    graphs = [H]
    print("graphs:", graphs)
    maps: list[dict] = []
    while graphs[-1].number_of_nodes() > coarsen_limit and len(graphs) < max_levels:
        Gc, label = coarsen_graph(graphs[-1], weight=weight, seed=trial_seed)
        if Gc.number_of_nodes() == graphs[-1].number_of_nodes():
            break
        graphs.append(Gc)
        maps.append(label)
    return graphs, maps

In [6]:
graphs, maps = coarsen_chain(G_K, trial_seed=42)

graphs: [<networkx.classes.graph.Graph object at 0x7f119ab66470>]


In [7]:
last_coarsed_graph, last_coarsed_map = graphs[-1], maps[-1]
print(f"K: |V|={last_coarsed_graph.number_of_nodes()}, |E|={last_coarsed_graph.number_of_edges()}")

K: |V|=69, |E|=548


In [8]:
from collections import defaultdict
import networkx as nx
import numpy as np
import neal

def build_qubo_from_graph(H: nx.Graph, *, balance_weight: float = 1.0, target_weight: float | None = None):
    """
    QUBO for balanced 2-way cut on subgraph H.
    Minimize: sum_{(u,v)∈E} w_uv * [x_u + x_v - 2 x_u x_v] + λ (sum_u c_u x_u - T)^2
    where c_u = node weight (vweight), T = target_weight (default = 0.5 * sum c_u).
    Returns Q (dict[(u,v)] -> coeff) upper-triangular including diagonals.
    """
    Q = defaultdict(float)
    nodes = list(H.nodes())
    c = {u: float(H.nodes[u].get('vweight', 1.0)) for u in nodes}
    Ctot = float(sum(c.values()))
    T = Ctot * 0.5 if target_weight is None else float(target_weight)

    # Edge XOR linear terms: + deg_w(u) * x_u
    degw = {u: 0.0 for u in nodes}
    for u, v, data in H.edges(data=True):
        w = float(data.get('weight', 1.0))
        degw[u] += w
        degw[v] += w

    # Diagonal: edge linear + balance diag: λ c_u^2
    for u in nodes:
        Q[(u, u)] += degw[u]
        Q[(u, u)] += balance_weight * (c[u] ** 2)
        # Linear from balance: -2 λ T c_u
        Q[(u, u)] += -2.0 * balance_weight * T * c[u] # THIS IS WRONG AT THE QUBO FORMULATION PDF, HERE IS CORRECTED

    # Off-diagonals:
    # Balance dense term: + 2 λ c_u c_v
    # Edge XOR quad term: -2 w_uv if edge exists
    # We add only for u < v (upper triangle)
    for i in range(len(nodes)):
        u = nodes[i]
        for j in range(i + 1, len(nodes)):
            v = nodes[j]
            coef = 2.0 * balance_weight * c[u] * c[v]
            if H.has_edge(u, v):
                coef += -2.0 * float(H[u][v].get('weight', 1.0))
            if coef != 0.0:
                Q[(u, v)] += coef

    return dict(Q)

def anneal_bipartition(H: nx.Graph, *, num_reads=200, balance_weight=1.0, target_weight: float | None = None, sampler=None):
    """Run simulated annealing on H’s 2-way QUBO and return part dict {node: 0|1} and best energy."""
    Q = build_qubo_from_graph(H, balance_weight=balance_weight, target_weight=target_weight)
    sampler = sampler or neal.SimulatedAnnealingSampler()
    resp = sampler.sample_qubo(Q, num_reads=num_reads)
    best = resp.first
    x = best.sample  # {node: 0/1}
    # normalize: ensure both labels present
    if all(bit == 0 for bit in x.values()):
        # flip a single boundary node to avoid empty part
        u0 = next(iter(x))
        x[u0] = 1
    return x, best.energy

def cut_value(H: nx.Graph, part: dict, weight='weight') -> float:
    cut = 0.0
    for u, v, data in H.edges(data=True):
        if part.get(u) != part.get(v):
            cut += float(data.get(weight, 1.0))
    return cut

In [9]:
def recursive_kway_anneal(Gc: nx.Graph, k: int, *, balance_weight=1.0, num_reads=200, choose_by='vweight'):
    """
    Perform k-way partition on Gc by repeatedly bisecting one block using the annealer.
    Returns dict {node: label in 0..k-1} for nodes in Gc.
    """
    if k <= 1:
        return {n: 0 for n in Gc.nodes()}
    # start with one block
    blocks: dict[int, set] = {0: set(Gc.nodes())}
    next_lbl = 1

    def block_weight(ns):
        if choose_by == 'vweight' and len(ns) > 0:
            return float(sum(Gc.nodes[n].get('vweight', 1.0) for n in ns))
        return float(len(ns))

    while len(blocks) < k:
        # pick a block to split
        bid = max(blocks, key=lambda b: block_weight(blocks[b]))
        nodes = blocks.pop(bid)
        H = Gc.subgraph(nodes).copy()
        # target to split H into halves by its own weight
        T = 0.5 * sum(H.nodes[n].get('vweight', 1.0) for n in H.nodes())
        x, _ = anneal_bipartition(H, num_reads=num_reads, balance_weight=balance_weight, target_weight=T)

        A = {n for n, bit in x.items() if bit == 0}
        B = set(H.nodes()) - A
        # guard against empty sets
        if len(A) == 0 or len(B) == 0:
            # fallback: naive split
            half = len(nodes) // 2
            A = set(list(nodes)[:half])
            B = set(nodes) - A
        blocks[bid] = A
        blocks[next_lbl] = B
        next_lbl += 1

    # flatten to labels
    out = {}
    for lbl, ns in blocks.items():
        for n in ns:
            out[n] = lbl
    # remap labels to 0..k-1
    remap = {lbl: i for i, lbl in enumerate(sorted(blocks.keys()))}
    return {n: remap[lbl] for n, lbl in out.items()}

In [10]:
def lift_partition_to_finer(graphs: list[nx.Graph], maps: list[dict], coarse_part: dict):
    """
    Given graphs[0]=original,...,graphs[L]=coarsest and maps[l] mapping fine(node)->coarse(node) for each l,
    lift coarse_part defined on graphs[L] back to original graph’s nodes.
    """
    part = coarse_part
    # traverse maps in reverse: level L-1 down to 0
    for l in range(len(maps) - 1, -1, -1):
        fine_G = graphs[l]
        label_map = maps[l]          # fine -> coarse
        # build fine partition by pulling labels from current part via coarse id
        fine_part = {}
        for u in fine_G.nodes():
            cu = label_map[u]
            fine_part[u] = part.get(cu, 0)
        part = fine_part
    return part
    

In [11]:
# Parameters
K_TARGET = 16                 # number of parts desired
BALANCE_LAMBDA = 1
NUM_READS = 1000

Gc = last_coarsed_graph          # coarsest graph
part_k_coarse = recursive_kway_anneal(
    Gc, K_TARGET,
    balance_weight=BALANCE_LAMBDA,
    num_reads=NUM_READS,
    choose_by='vweight'
)

# Lift back to original nodes
part_k_orig = lift_partition_to_finer(graphs, maps, part_k_coarse)

# Summarize on original graph
summarize_generic(G_K, part_k_orig, f'K [quantum k={K_TARGET}]')

# Optional: compute cut on coarsest, too
print("Coarsest cut:", cut_value(Gc, part_k_coarse))

K [quantum k=16]: cut=31881.1804, parts=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], per=[2103.256376645351, 2105.7457738284197, 2093.2608990209415, 2106.831315116913, 2102.0017744724173, 2103.5680974361585, 2120.5454443148687, 2100.004023313338, 2122.456591478826, 2105.089658404142, 2107.1685813592317, 2086.0631655884554, 2097.3868283169577, 2100.241230103615, 2095.924093066491, 2096.3211213757263], total=33645.8650
Per difference: Max per: Min per: 36.39342589037051 2122.456591478826 2086.0631655884554
Coarsest cut: 31881.18040942603


In [12]:
# Baseline METIS comparison for the same K_TARGET
metis_part_K = partition_graph_metis(G_K, nparts=K_TARGET, weight='weight', seed=42, verbose=0)
summarize_generic(G_K, metis_part_K, f"METIS K [{K_TARGET}]")

METIS K [16]: cut=18127.3360, parts=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], per=[2101.324209690338, 2052.146135341699, 2063.166288877349, 2066.5010701710703, 2130.5845741601897, 2083.0383005070744, 2095.1148593938683, 2155.3410775614243, 2085.2365617989162, 2159.6697992685804, 2069.491596803579, 2110.1608488051465, 2164.1929841559913, 2093.0098363468087, 2054.9147893267277, 2161.9720416330892], total=33645.8650
Per difference: Max per: Min per: 112.04684881429239 2164.1929841559913 2052.146135341699
